# v16a — join the v14 and v15 caches (no new GPU time)

v15 falsified its own premise but left behind something v14 never had: **tier-2 hard negatives**
— a benign imperative sentence inserted into an untrusted document at the same position,
length-matched to the payload it replaces.

v15's second finding was that the detector fires on those at **40.75–52.39%**, against 19–20% on
the untouched carrier and 60–71% on real injections. So it is substantially detecting *that an
instruction is present in untrusted content*, not *that the instruction is malicious*.

**But it was never shown a benign instruction during training.** That is what v16 tests.

## Why this costs no extraction

| source | provides | rows |
|---|---|---|
| `representation_cache/unified` (v14) | last-token states, direct families + all 8 benign corpora + Dolly baseline | 9,910 + 2,000 |
| `representation_cache/chunked_v15` | **final chunk = the last token**, indirect families **and the hard negatives** | 3,335 |

The non-indirect rows are byte-identical between the two corpora (9,910/9,910 exact text match),
and both caches were produced by the same model, system prompt, `max_len` and layer set — checked
below, not assumed. v15's final chunk closes on the last real token by construction, so it *is*
v14's feature.

## The grouping problem, and why `payload_id` is no longer enough

An injected row and its hard-negative twin **share a carrier**. If one lands in train and the
other in test, the model can recognise the document instead of the payload — a new leak that
v14's `payload_id` grouping does not close, because a BIPIA payload also spans three carriers.

The obvious fix — the connected component of the (`payload_id`, `pair_id`) graph — turns out to
be unusable where it matters most:

| contrast | components | largest |
|---|---|---|
| BIPIA + BIPIA-HardNeg | 51 | 317 rows (18.9%) |
| InjecAgent + InjecAgent-HardNeg | **1** | **695 rows (100%)** |

InjecAgent has 17 templates and 62 attacker instructions, each appearing across several
templates, so its graph is fully connected. Under component grouping **every InjecAgent row lands
in one fold** — which silently turns Exp 1 into leave-one-family-out for InjecAgent and makes its
numbers incomparable with v14's.

So v16a emits **two** keys and v16b states which it used:

- **`group_carrier`** (primary) — `pair_id` for indirect rows, `payload_id` elsewhere. Closes the
  new twin leak, and keeps the *same* payload-across-carriers leakage profile v14 had, so v16
  stays comparable with v14.
- **`group_component`** (sensitivity) — the union-find key. Stricter, used where it yields enough
  groups (BIPIA), reported as a robustness check rather than the headline.

In [ ]:
import os, json, gc
import numpy as np, pandas as pd

V14_CACHE = "representation_cache/unified"
V15_CACHE = "representation_cache/chunked_v15"
OUT       = "representation_cache/joined_v16"
os.makedirs(OUT, exist_ok=True)

IND = ["BIPIA", "BIPIA-Clean", "BIPIA-HardNeg",
       "InjecAgent", "InjecAgent-Clean", "InjecAgent-HardNeg"]

m14 = json.load(open(f"{V14_CACHE}/manifest.json"))
m15 = json.load(open(f"{V15_CACHE}/manifest.json"))

# Commensurability is CHECKED, not assumed. A mismatch in any of these would make the two
# halves of the joined matrix incomparable, and the failure would be silent.
for k in ["model", "system_prompt", "max_len", "layers"]:
    assert m14[k] == m15[k], f"cache mismatch on {k}: {m14[k]!r} vs {m15[k]!r}"
LAYERS = m14["layers"]
MC     = m15["max_chunks"]
print(f"caches agree on model / system_prompt / max_len / layers")
print(f"  layers {LAYERS}")

v14 = pd.read_csv("datasets/unified/injection_corpus.csv").reset_index(drop=True)
v15 = pd.read_csv("datasets/unified/injection_corpus_v15.csv").reset_index(drop=True)
r15 = pd.read_csv(f"{V15_CACHE}/rows.csv").reset_index(drop=True)
print(f"  v14 corpus {len(v14):,} | v15 corpus {len(v15):,} | v15 chunk cache {len(r15):,}")

In [ ]:
# ---- row plan: where each v16 row's state comes from ------------------------
# v15's corpus is the target row set. Indirect rows (and the hard negatives) come from the
# chunk cache's final chunk; everything else comes from v14's last-token cache, matched by
# EXACT TEXT -- the two corpora share those rows byte for byte.
v16 = v15.copy()
v16["src"] = np.where(v16.corpus.isin(IND), "v15", "v14")

# v14 lookup by text. Duplicate texts would make the mapping ambiguous; assert they are gone.
assert not v14.text.duplicated().any(), "duplicate texts in v14 corpus -- text join is unsafe"
t14 = pd.Series(np.arange(len(v14)), index=v14.text.values)
# v15 chunk-cache lookup by text (r15 carries its own row order)
assert not r15.text.duplicated().any(), "duplicate texts in v15 chunk cache"
t15 = pd.Series(np.arange(len(r15)), index=r15.text.values)

need14 = v16.src.values == "v14"
need15 = ~need14
src_row = np.full(len(v16), -1, np.int64)
src_row[need14] = t14.reindex(v16.text.values[need14]).values
src_row[need15] = t15.reindex(v16.text.values[need15]).values
missing = int((src_row < 0).sum())
print(f"rows from v14 cache: {int(need14.sum()):,} | from v15 cache: {int(need15.sum()):,}"
      f" | unresolved: {missing}")
assert missing == 0, "some rows have no state in either cache"
v16["src_row"] = src_row
# n_chunks is needed to find the last real chunk in the v15 cache
v16["n_chunks"] = 0
v16.loc[need15, "n_chunks"] = r15.n_chunks.values[src_row[need15]]

In [ ]:
# ---- group keys: TWO of them, and the choice matters ------------------------
# The new leak in v16: an injected row and its hard-negative twin share a carrier. If one
# is in train and the other in test, the model can recognise the DOCUMENT instead of the
# payload -- which would make arm B look good for the wrong reason.
#
# The obvious fix is the connected component of the (payload_id, pair_id) graph, closing
# both the carrier and the payload route. Measured on the real data, that is not usable:
#
#   BIPIA + BIPIA-HardNeg          : 51 components, largest  317 rows (18.9%)  -- fine
#   InjecAgent + InjecAgent-HardNeg:  1 component,  largest  695 rows (100%)   -- unusable
#
# InjecAgent has only 17 templates and 62 attacker instructions, every instruction appearing
# across several templates, so the graph is fully connected. Under component grouping ALL
# InjecAgent rows land in one fold, which silently turns Exp 1 into leave-one-family-out for
# InjecAgent and makes its numbers incomparable with v14's. That is exactly the kind of
# invisible change of question this project has been bitten by before.
#
# So: TWO keys, both emitted, and v16b states which it used.
#   group_carrier  (PRIMARY) -- pair_id for indirect rows, payload_id elsewhere.
#                   Closes the NEW twin leak. Keeps the SAME payload-across-carriers
#                   leakage profile v14 had, so v16 stays comparable with v14.
#   group_component (SENSITIVITY) -- the union-find key. Stricter, but only usable where
#                   it yields more than a handful of groups.
par = {}
def find(x):
    par.setdefault(x, x)
    while par[x] != x: par[x] = par[par[x]]; x = par[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: par[ra] = rb

is_ind = v16.corpus.isin(IND).values
pid = v16.payload_id.astype(str).values
cid = v16.pair_id.astype(str).values
has_cid = np.array([bool(c) and c != "nan" for c in cid])

for j in range(len(v16)):
    a = f"P:{pid[j]}"
    union(a, f"C:{cid[j]}" if has_cid[j] else a)
v16["group_component"] = [find(f"P:{p}") for p in pid]
v16["group_carrier"]   = np.where(is_ind & has_cid, "C:" + cid, "P:" + pid)

print("group keys:")
for k in ["group_carrier", "group_component"]:
    print(f"  {k:<16} {v16[k].nunique():>6,} groups over {len(v16):,} rows")
print("\n  per contrast, largest group as a share of that contrast:")
for pair in [("BIPIA", "BIPIA-HardNeg"), ("InjecAgent", "InjecAgent-HardNeg")]:
    s = v16[v16.corpus.isin(pair)]
    for k in ["group_carrier", "group_component"]:
        g = s.groupby(k).size()
        flag = "  <- unusable for 5-fold" if g.max()/len(s) > 0.5 else ""
        print(f"    {pair[0]:<11} {k:<16} {len(g):>4} groups, largest "
              f"{g.max():>4} ({g.max()/len(s)*100:5.1f}%){flag}")

def tier(c):
    if c.endswith("HardNeg"): return "hardneg"
    if c.endswith("Clean"):   return "clean"
    if c == "Baseline-Dolly": return "baseline"
    return "inj" if c in ("BIPIA","InjecAgent","SPML","DeepSet") else "benign"
v16["tier"] = v16.corpus.map(tier)
print("\n", v16.groupby(["tier","label"]).size().unstack(fill_value=0).to_string(), sep="")

In [ ]:
# ---- build the joined last-token matrix, one layer at a time ----------------
SH14 = sorted(f for f in os.listdir(V14_CACHE) if f.startswith("shard_"))
SH15 = sorted(f for f in os.listdir(V15_CACHE) if f.startswith("shard_"))
I14  = [np.load(f"{V14_CACHE}/{f}")["row_idx"] for f in SH14]
I15  = [np.load(f"{V15_CACHE}/{f}")["row_idx"] for f in SH15]

src_row = v16.src_row.values
n15    = v16.n_chunks.values

for li in LAYERS:
    X = np.zeros((len(v16), 4096), np.float32)
    # v14 side: already a last-token vector
    A = np.zeros((len(v14), 4096), np.float32)
    for f, idx in zip(SH14, I14):
        A[idx] = np.load(f"{V14_CACHE}/{f}")[f"layer_{li}"]
    sel = np.where(need14)[0]
    X[sel] = A[src_row[sel]]
    del A
    # v15 side: take the LAST VALID CHUNK, which closes on the final real token
    B = np.zeros((len(r15), MC, 4096), np.float16)
    for f, idx in zip(SH15, I15):
        B[idx] = np.load(f"{V15_CACHE}/{f}")[f"layer_{li}"]
    sel = np.where(need15)[0]
    X[sel] = B[src_row[sel], np.maximum(0, n15[sel] - 1)].astype(np.float32)
    del B; gc.collect()
    assert np.isfinite(X).all(), f"non-finite states at layer {li}"
    np.save(f"{OUT}/layer_{li}.npy", X)
    print(f"  layer {li:>2}: {X.shape} float32  ({X.nbytes/1e9:.2f} GB)  "
          f"norm med {np.median(np.linalg.norm(X,axis=1)):.1f}")
    del X; gc.collect()

In [ ]:
# ---- sanity: do the two halves live in the same space? ---------------------
# If v14's and v15's states were on different scales or offsets, PCA and Mahalanobis would
# silently encode "which cache did this row come from". The direct and indirect families are
# genuinely different text, so their norms need not match -- but they must be the same ORDER,
# and the two sources' BENIGN rows should overlap.
li = LAYERS[0]
X = np.load(f"{OUT}/layer_{li}.npy")
nm = np.linalg.norm(X, axis=1)
print(f"state-norm by source and tier at layer {li}:")
tmp = v16.assign(norm=nm)
print(tmp.groupby(["src","tier"]).norm.agg(["count","median","min","max"]).round(1).to_string())
q14 = np.percentile(nm[need14], [5, 50, 95])
q15 = np.percentile(nm[need15], [5, 50, 95])
print(f"\n  v14-sourced norms p5/50/95: {q14.round(1)}")
print(f"  v15-sourced norms p5/50/95: {q15.round(1)}")
ratio = q15[1] / q14[1]
print(f"  median ratio {ratio:.3f}"
      + ("  <- OK, same order" if 0.5 < ratio < 2.0 else "  <- SUSPICIOUS, investigate before trusting v16b"))

v16.drop(columns=["text"]).to_csv(f"{OUT}/rows_v16.csv", index=False)
json.dump(dict(model=m14["model"], layers=LAYERS, system_prompt=m14["system_prompt"],
               max_len=m14["max_len"], rows=int(len(v16)),
               sources={"v14": int(need14.sum()), "v15_last_chunk": int(need15.sum())},
               groups_carrier=int(v16.group_carrier.nunique()),
               groups_component=int(v16.group_component.nunique()),
               group_key="group_carrier (primary): pair_id for indirect rows, payload_id elsewhere. group_component (sensitivity): union-find over (payload_id, pair_id).",
               note="last-token states; v15 rows use the final valid chunk"),
          open(f"{OUT}/manifest.json", "w"), indent=2)
print(f"\nwrote {OUT}/ -- rows_v16.csv, manifest.json, layer_*.npy")
print("next: v16b_analysis.ipynb")